In [21]:
import pandas as pd

# Documentation
Acesse a documentação clicando [aqui](https://pandas.pydata.org/docs/)

# Funcionamento do Pandas

Muita coisa do **pandas** é feito utilizando `Cython` (código C otimizado que funciona no Python). Muita da performance do pandas vem de partes escritas em Cython, que compila código Python otimizado em C. Ele é usado principalmente nas partes mais críticas — como operações com índices, agrupamentos, merges e leitura de arquivos — enquanto o restante da API é implementado em Python puro. Ela permite que você escreva código que combina a sintaxe simples do Python com a performance do C.
```python
def calcular(int n):
    cdef int resultado = 0
    cdef int i
    for i in range(n):
        resultado += i * i
    return resultado
```
E depois pode rodar no shell:
```shell
cythonize -i meu_arquivo.pyx
```
As *colunas* (`Series`) do pandas são essencialmente arrays do **NumPy** (em memória) com index. Isso significa que, operações *vetorizadas* como:
```python
df['x'] + df['y']
```
São executadas em **C**, dentro do NumPy, sem loop.

Conceitualmente, um DataFrame parece um dicionário de Series (uma por coluna).
Internamente, porém, ele usa estruturas chamadas `BlockManager` para armazenar colunas em blocos otimizados na memória — o que permite operações vetorizadas rápidas e economia de espaço.
Conceitualmente um `DataFrame` se parece com um dicionário de `Series`: 
```python
# Estrutura de um DataFrame
df = {
    'coluna_a': pd.Series([...]),
    'coluna_b': pd.Series([...]),
    ...,
    'coluna_z': pd.Series([...])
}
```

In [29]:
df = pd.DataFrame(
    {
        "col_int_1": [1, 2, 3],
        "col_int_2": [4, 5, 6],
        "col_string": ['a', 'b', 'c'],
        "col_float": [1.8, 0.19, 0.15]
    }
)
print(type(df._mgr))

<class 'pandas.core.internals.managers.BlockManager'>


O `BlockManager` armazena os dados de um `DataFrame` em blocos organizados por dtype, como podemos ver abaixo:

In [30]:
df._mgr.blocks

(NumpyBlock: slice(0, 2, 1), 2 x 3, dtype: int64,
 NumpyBlock: slice(2, 3, 1), 1 x 3, dtype: object,
 NumpyBlock: slice(3, 4, 1), 1 x 3, dtype: float64)

O `BlockManager` tem mantém a referência para o nome das colunas, posição das colunas dentro dos blocos e o Index do `DataFrame`. Por conta dessa estrutura ele entrega **alta performance** através de vetorização, slicing, atribuição, concatenação, merge - tudo isso é otimizado por operar em `blocos` contínuos, não coluna a coluna.  

In [32]:
# inspecionando um block
for block in df._mgr.blocks:
    print("----")
    print(type(block))
    print("dtype:", block.dtype)
    print("shape:", block.shape)
    print("colunas:", df.columns[block.mgr_locs])

----
<class 'pandas.core.internals.blocks.NumpyBlock'>
dtype: int64
shape: (2, 3)
colunas: Index(['col_int_1', 'col_int_2'], dtype='object')
----
<class 'pandas.core.internals.blocks.NumpyBlock'>
dtype: object
shape: (1, 3)
colunas: Index(['col_string'], dtype='object')
----
<class 'pandas.core.internals.blocks.NumpyBlock'>
dtype: float64
shape: (1, 3)
colunas: Index(['col_float'], dtype='object')


In [35]:
# acessando o nosso primeiro bloco (colunas int)
df._mgr.blocks[0].values

array([[1, 2, 3],
       [4, 5, 6]])

In [37]:
# acessando os outros blocos
df._mgr.blocks[1].values

array([['a', 'b', 'c']], dtype=object)

In [38]:
# acessando os outros blocos
df._mgr.blocks[2].values

array([[1.8 , 0.19, 0.15]])

Como podemos ver tudo são `arrays` do `numpy` estruturados em blocos otimizados.

In [39]:
df._mgr

BlockManager
Items: Index(['col_int_1', 'col_int_2', 'col_string', 'col_float'], dtype='object')
Axis 1: RangeIndex(start=0, stop=3, step=1)
NumpyBlock: slice(0, 2, 1), 2 x 3, dtype: int64
NumpyBlock: slice(2, 3, 1), 1 x 3, dtype: object
NumpyBlock: slice(3, 4, 1), 1 x 3, dtype: float64

O `BlockManager` age da seguinte maneira:
```scss
DataFrame
 ├── Index  (labels das linhas)
 ├── Columns (labels das colunas)
 └── Blocks (agrupados por dtype)
        ├── FloatBlock: matriz contígua 2D
        ├── IntBlock: matriz contígua 2D
        └── ObjectBlock: matriz contígua 2D

```